In [26]:
%pip install srai[osm]

zsh:1: no matches found: srai[osm]
Note: you may need to restart the kernel to use updated packages.


In [13]:
from srai.loaders import OSMOnlineLoader
from srai.plotting import plot_regions
from srai.regionalizers import geocode_to_region_gdf
import geopandas as gp
import pandas

#query = {"power": "line"}
area = geocode_to_region_gdf("USA")
loader = OSMOnlineLoader()

#parks_gdf = loader.load(area, query)
parks_gdf = gp.read_file("Data/Electric_Power_Transmission_Lines.geojson")
nationalLinesBuffers = parks_gdf.buffer(0.043, resolution=16, cap_style='round', join_style='round', mitre_limit=5.0, single_sided=False)
nationalLinesBuffers.to_crs('EPSG:4326')
nationalBuffersGDF = gp.GeoDataFrame(nationalLinesBuffers, geometry=gp.GeoSeries(nationalLinesBuffers))
folium_map = plot_regions(area, colormap=["rgba(0,0,0,0)"], tiles_style="CartoDB positron")
nationalBuffersGDF.explore(m=folium_map, color="forestgreen")


/var/folders/8x/34jx632j291dnr4g1mv5z0kh0000gn/T/ipykernel_12744/2344972647.py:13: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  nationalLinesBuffers = parks_gdf.buffer(0.043, resolution=16, cap_style='round', join_style='round', mitre_limit=5.0, single_sided=False)
/Users/henryadams/Library/Python/3.9/lib/python/site-packages/pandas/core/dtypes/cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])


In [20]:
from srai.regionalizers import H3Regionalizer, geocode_to_region_gdf

regionalizer = H3Regionalizer(resolution=3)
regions = regionalizer.transform(area)

folium_map = plot_regions(area, colormap=["rgba(0,0,0,0.1)"], tiles_style="CartoDB positron")
plot_regions(regions_gdf=regions, map=folium_map)
regions.head()

,geometry
region_id,
8326edfffffffff,"POLYGON ((-93.53712 35.70186, -93.03813 36.176..."
832996fffffffff,"POLYGON ((-110.70159 36.24671, -110.25748 36.8..."
83299cfffffffff,"POLYGON ((-114.38213 37.96395, -113.95317 38.5..."
8328c6fffffffff,"POLYGON ((-123.61371 46.46249, -123.21854 46.9..."
83281afffffffff,"POLYGON ((-122.06974 43.02388, -121.67659 43.5..."


In [23]:
from srai.loaders import OSMPbfLoader

loader = OSMPbfLoader()
query = {'power': 'line'}
power_lines = loader.load(area, query)

ImportError: Missing optional dependency "quackosm". Please install required packages using `pip install srai[osm]`.

In [ ]:
from srai.embedders import CountEmbedder
from srai.joiners import IntersectionJoiner
from srai.loaders import OSMOnlineLoader
from srai.plotting import plot_regions, plot_numeric_data
from srai.regionalizers import H3Regionalizer, geocode_to_region_gdf
import osmnx as ox

nationalBuffersGDF.index.name = 'feature_id'

loader = OSMOnlineLoader()
joiner = IntersectionJoiner()
joint = joiner.transform(regions, nationalBuffersGDF)

embedder = CountEmbedder()
embeddings = embedder.transform(regions, nationalBuffersGDF, joint)

folium_map = plot_regions(area, colormap=["rgba(0,0,0,0.1)"], tiles_style="CartoDB positron")
plot_numeric_data(regions, "power_line", embeddings, map=folium_map)